# PARA-1a 探索：空白の揺れ（E1・E2、CR-1a のデータ・CPU・API なし）

**事後の探索です（ラベルは出しません）。** 決まりは `notes/para1a_explore_ws_plan.md`（データの前に書いたもの）。凍結した CR-1a・PARA-1a の結果は変えません。

- **セル1**：決まった版のコードと、NLTK・WordNet・stopwords（凍結の記録と同じもの）を入れます。
- **セル2**：G1・G2 のファイルをハッシュで確かめたあと、まず**元の数え方で G3・PARA-1a の数がそのまま出ること**を確かめます（違えば止まります）。そのあと E1（表記ゆれ・空白だけの違いを除いた A2）と E2（空白だけ違う続きを「そのまま書いた」側に数え直した d_s など）を計算します。
- 問題文・応答・語の中身はどこにも表示しません。結果は数だけで、Drive の `DECISION_POINTS/cr1a/PARA1A/PARA1A_explore_ws_result.json` に入ります。
- 費用 $0。シークレットは `GITHUB_TOKEN` だけ。CPU のランタイムで「すべてのセルを実行」。15〜30 分くらい。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, pinned NLTK code and data, notifications.  No API key.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, urllib.request, zipfile
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
STUDY = "CR-1a"
EXPECTED_HEAD = "98cd63a61f913ff9c9d2023bb25d5509bb64c3e7"
G1_HEAD = "7d95ee90a7b536fb85326e2a30981b9bc8030421"
G2_HEAD = "f6c6bb94fe2ff77a5726e71b835f7ee0632a9d4d"
G1_ITEMS_SHA = "bd42639245cb234f86e0bcd97092db67ae2842c9f1e48e123a9ee1b484c931b5"
G2_SUMMARY_SHA = "17afa043db733289e49eb4d55ac1c160760067f36eb2a9c4cdfa36e956f159f1"
G3_RESULT_SHA = "4addb0ed40b914547e272b1c450285425ea9cf8c76f81b1c28576b127c8217a7"
PARTA_RESULT_SHA = "6d3873e23e3a2b1c4d1f6edf23c70bdd4f2a89247aa7b6e20f0a83be6f0e43f2"
PARA1_FREEZE, PARA1A_FREEZE, LOCK3 = "69dd041", "7d95ee9", "69b43d8"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
def is_anc(a, b):
    return subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", a, b]).returncode == 0
for anc in (PARA1_FREEZE, PARA1A_FREEZE, LOCK3, G1_HEAD, G2_HEAD, "bcafae7", "bbd6b4a", "e0ac94a", "d7a4e8b"):
    assert is_anc(anc, head), anc + " が祖先ではありません"
TIMING = {"para1_freeze": PARA1_FREEZE, "para1a_freeze": PARA1A_FREEZE, "lock3": LOCK3,
          "para1_freeze_is_ancestor_of_lock3": is_anc(PARA1_FREEZE, LOCK3),
          "para1a_freeze_is_ancestor_of_lock3": is_anc(PARA1A_FREEZE, LOCK3)}
assert TIMING["para1_freeze_is_ancestor_of_lock3"] and TIMING["para1a_freeze_is_ancestor_of_lock3"]
token = None
env.pop("GITHUB_TOKEN", None)
sys.path.insert(0, str(REPO))
from pork import para1a_parta as P

# NLTK code: the commit used before the freeze.  Data: nltk_data gh-pages at the recorded commit, SHA256-checked.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                f"nltk @ git+https://github.com/nltk/nltk@{P.NLTK_COMMIT}"], check=True)
NLTK_DIR = Path("/content/nltk_data_pinned")
(NLTK_DIR / "corpora").mkdir(parents=True, exist_ok=True)
for name, want in (("wordnet", P.WORDNET_ZIP_SHA), ("stopwords", P.STOPWORDS_ZIP_SHA)):
    z = NLTK_DIR / "corpora" / (name + ".zip")
    url = f"https://raw.githubusercontent.com/nltk/nltk_data/{P.NLTK_DATA_COMMIT}/packages/corpora/{name}.zip"
    z.write_bytes(urllib.request.urlopen(url, timeout=120).read())
    got = hashlib.sha256(z.read_bytes()).hexdigest()
    assert got == want, name + ".zip の SHA256 が凍結の記録と違います。止めます。"
    shutil.rmtree(NLTK_DIR / "corpora" / name, ignore_errors=True)
    zipfile.ZipFile(z).extractall(NLTK_DIR / "corpora")
os.environ["NLTK_DATA"] = str(NLTK_DIR)
import nltk
nltk.data.path[:] = [str(NLTK_DIR)]
from nltk.corpus import wordnet as wn, stopwords
STOP = stopwords.words("english")
assert hashlib.sha256((NLTK_DIR / "corpora/stopwords/english").read_bytes()).hexdigest() == P.STOPWORDS_EN_FILE_SHA
assert len(STOP) == 198 and hashlib.sha256("\n".join(STOP).encode("utf-8")).hexdigest() == P.STOPWORDS_LIST_SHA
assert wn.get_version() == "3.0"
import numpy
VERSIONS = {"nltk": nltk.__version__, "nltk_commit": P.NLTK_COMMIT, "nltk_data_commit": P.NLTK_DATA_COMMIT,
            "wordnet": wn.get_version(), "stopwords_en": len(STOP), "numpy": numpy.__version__,
            "python": sys.version.split()[0], "cpus": os.cpu_count()}
print("exact head:", head, "|", VERSIONS)
OUT = Path("/content/drive/MyDrive/DECISION_POINTS/cr1a/PARA1A")
OUT.mkdir(parents=True, exist_ok=True)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'PARA-1a 探索 空白の揺れ E1・E2（CR-1a・CPU・API なし）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - PARA-1a explore whitespace', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — integrity checks, reproduce G3 / Part A numbers, then E1 and E2.  Numbers only.
t1 = time.time()
import numpy as np
from pork import cr1_run as R
from pork import cr1_requests as RQ
from pork import cr1_analysis as A
from pork import cr1_g3 as G3
from pork import cr1_items as CI
from pork import para1a_explore as E
g3_text = (REPO / "reports/cr1a_g3_result.json").read_text()
assert hashlib.sha256(g3_text.encode("utf-8")).hexdigest() == G3_RESULT_SHA, "G3 の結果ファイルが違います。止めます。"
pa_text = (REPO / "reports/para1a_partA_result.json").read_text()
assert hashlib.sha256(pa_text.encode("utf-8")).hexdigest() == PARTA_RESULT_SHA, "部A の結果ファイルが違います。止めます。"
g3 = json.loads(g3_text); pa = json.loads(pa_text)
rows, DATA_ID = R.load_rows_public()
chain = CI.chain_and_candidates(rows)
del rows
assert CI.candidates_sha256(chain) == R.CANDIDATES_SHA256, "候補の一覧が段2 と違います。止めます。"
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr1a")
ck = json.loads((D / "G1/CR1A_G1_checkpoint_PRIVATE.json").read_text())
assert ck["header"] == R.header(STUDY, G1_HEAD, R.CANDIDATES_SHA256, A.MODEL, RQ.SDK_VERSION)
R.validate(ck, chain)
g1 = R.summarize_g1(ck)
assert g1["verdict"] == "GO" and g1["final_items_ids_sha256"] == G1_ITEMS_SHA
s2_text = (D / "G2/CR1A_G2_summary.json").read_text()
assert hashlib.sha256(s2_text.encode("utf-8")).hexdigest() == G2_SUMMARY_SHA, "G2 の記録が変わっています。止めます。"
g2 = json.loads(s2_text)
final = g1["final_items"]
res, load_stats = G3.load_results(D / "G2", g2["batch_records"], final)
by_item = G3.per_item(res, len(final))
order = P.item_order(final)
items_e, items_w, pitems = [], [], []
for i, fi in enumerate(final):
    p = ck["items"][str(fi["template"])]
    assert hashlib.sha256(p["text"].encode("utf-8")).hexdigest() == fi["text_sha256"]
    ie = E.build_item_rule(i, int(fi["template"]), p["text"], by_item[i], "exact")
    items_e.append(ie)
    items_w.append(E.build_item_rule(i, int(fi["template"]), p["text"], by_item[i], "ws"))
    if not ie.missing.any():
        pitems.append(P.extract_item(i, int(fi["template"]), order[i], p["text"], by_item[i], u=ie.u))
assert [it.index for it in pitems] == g3["main_item_indices"]
del res, by_item
print("読み込みと確かめ OK |", round(time.time() - t1), "秒")

# E2 first (CR-1a numbers); the exact rule must reproduce G3.
e2 = E.e2(items_e, items_w, len(final), g2["unresolved"])
mr_g3, mr_ex = g3["must_report"], e2["exact"]["must_report"]
def close(a, b):
    return abs(a - b) <= 1e-12
assert close(mr_ex["main0"]["value"], mr_g3["main0"]["value"]) and mr_ex["main0"]["ci"] == mr_g3["main0"]["ci"]
assert close(mr_ex["positive_ds_share"], mr_g3["positive_ds_share"]) and mr_ex["main1_target_items"] == mr_g3["main1_target_items"]
assert close(mr_ex["n_diff_zero_share"], mr_g3["n_diff_zero_share"]) and mr_ex["label_info"] == mr_g3["label_info"]
assert e2["exact"]["would_be_verdict_descriptive_only"] == g3["verdict"]
print("E2：元の数え方で G3 の数を再現 OK |", round(time.time() - t1), "秒")

# E1 (Part A A2 on subsets); the full set must reproduce the Part A numbers.
T = P.Typing(P.load_typing_module(REPO / "scripts/para1_typing.py", "para1_typing_main"), wn, STOP)
e1 = E.e1(pitems, T)
full = e1["all_as_partA"]
assert full["A1"]["substitutions"] == pa["part_a"]["A1"]["substitutions"]
for name in ("SEMANTIC", "FORM", "BOTH"):
    assert close(full["A2"][name]["excess"]["value"], pa["part_a"]["A2"][name]["excess"]["value"]), name
    assert full["A2"][name]["excess"]["ci"] == pa["part_a"]["A2"][name]["excess"]["ci"], name
print("E1：除かないときに部A の数を再現 OK |", round(time.time() - t1), "秒")

out = {"study": "PARA-1a / CR-1a exploration (post-hoc, no labels)", "plan": "notes/para1a_explore_ws_plan.md",
       "reproduced": {"G3_must_report": True, "partA_A2": True}, "E1": e1, "E2": e2, "timing": TIMING,
       "inputs": {"code_head": head, "g3_result_sha256": G3_RESULT_SHA, "partA_result_sha256": PARTA_RESULT_SHA,
                  "dataset": DATA_ID}, "versions": VERSIONS, "seconds": round(time.time() - t1, 1)}
text = A.result_summary(out)
tmp = OUT / "PARA1A_explore_ws_result.json.tmp"
tmp.write_text(text)
os.replace(tmp, OUT / "PARA1A_explore_ws_result.json")
for nm, v in e1.items():
    print(nm, "| 置き換え", v["A1"]["substitutions"], "| SEMANTIC の超過", round(v["A2"]["SEMANTIC"]["excess"]["value"], 5),
          [round(x, 5) for x in v["A2"]["SEMANTIC"]["excess"]["ci"]])
for nm in ("exact", "ws"):
    m = e2[nm]["must_report"]
    print(nm, "| d_s>0 の割合", round(m["positive_ds_share"], 4), "| 主1 の対象", m["main1_target_items"],
          "| 主0", round(m["main0"]["value"], 5))
print("移った続き:", e2["moved_to_same_by_ws_rule"], "| 時間:", out["seconds"], "秒")
notify('終わりました：PARA-1a 探索（空白の揺れ）の結果を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()